# Qwen Automatic Evaluation

In [ ]:

from pathlib import Path
import json
import os
import random

import nltk
import numpy as np
import pandas as pd
import torch
from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from tqdm.notebook import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

SEED = 42
MAX_INPUT_TOKENS = 12000
MAX_NEW_TOKENS = 128
DATASET_FILE = Path("evaluation_dataset.jsonl")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

smooth = SmoothingFunction().method1
rouge_scorer_instance = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True,
)


def load_dataset():
    records = []
    with DATASET_FILE.open("r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def format_context(record):
    commit_text = "\n".join(
        f"{item['hash']} | {item['date']} | {item['subject']}"
        for item in record["commits"]
    )

    if not commit_text:
        commit_text = "(none)"

    return f"""Repository: {record['repository']}
Repository Commit: {record['repository_sha']}
File: {record['file_name']}

Code:
{record['source_code']}

Code Size:
Lines: {record['code_size']['lines']}
Nonblank Lines: {record['code_size']['nonblank_lines']}
Characters: {record['code_size']['characters']}
Bytes: {record['code_size']['bytes']}
NLOC: {record['code_size']['nloc']}

Comments:
{record['comments'] or '(none)'}

README ({record['readme_name'] or 'none'}):
{record['readme_content'] or '(none)'}

Complexity:
Function Count: {record['complexity']['function_count']}
Mean Cyclomatic Complexity: {record['complexity']['mean_ccn']}
Maximum Cyclomatic Complexity: {record['complexity']['max_ccn']}

File Commit History:
{commit_text}
"""


def prompt_zero(code, readme, complexity, comments, commits):
    return f"""
You are a senior software engineer with expertise in code readability, maintainability, and program comprehension. Given a source code file along with contextual information, generate a clear and concise code summary. Do not repeat the code. Ensure the summary is optimized for the following assessment aspects: Semantic Correctness, Informativeness, Readability, Usefulness, and Overall Quality.
1. Semantic Correctness: The summary must accurately represent the code's logic, control flow, and technical behavior without misinterpretation.
2. Informativeness: The summary must include key details such as the primary objective, major input/output, and relevant dependencies.
3. Readability: The summary must use straightforward, structured language that is easy for other developers to comprehend.
4. Usefulness: The summary must provide context that aids in code maintenance and program understanding.
5. Overall Quality: The summary must be professional, concise, and adhere to industry-standard documentation practices.

Inputs:
- Code: {code}
- README: {readme}
- Complexity: {complexity}
- Comments: {comments}
- Commits: {commits}
- Code Size: {len(code)}

Output ONLY the final summary.

"""

def prompt_few(code, readme, complexity, comments, commits):
    return f"""
ou are a senior software engineer with expertise in code readability, maintainability, and program comprehension. Given a source code file along with contextual information, generate a clear and concise code summary. Do not repeat the code. Ensure the summary is optimized for the following assessment aspects: Semantic Correctness, Informativeness, Readability, Usefulness, and Overall Quality.
1. Semantic Correctness: The summary must accurately represent the code's logic, control flow, and technical behavior without misinterpretation.
2. Informativeness: The summary must include key details such as the primary objective, major input/output, and relevant dependencies.
3. Readability: The summary must use straightforward, structured language that is easy for other developers to comprehend.
4. Usefulness: The summary must provide context that aids in code maintenance and program understanding.
5. Overall Quality: The summary must be professional, concise, and adhere to industry-standard documentation practices.
Below are examples of how to generate the summary:
### EXAMPLE 1 ###
Inputs:
- Code: 
def load_config(filepath):
   if not os.path.exists(filepath):
   raise FileNotFoundError("Config missing")
   with open(filepath, 'r') as 
f: return json.load(f)
- README: Utility to initialize system settings.
- Complexity: Cyclomatic Complexity: 2
- Comments: Checks for file existence before loading JSON.
- Commits: Added error handling for missing configuration files.
- Code Size: 5 lines
Output:
The `load_config` function acts as a secure utility to initialize system settings by parsing a JSON configuration file. Its primary objective is to safely read the file and return a dictionary of settings (Output) based on the provided `filepath` string (Input). The function includes a critical validation step—added in a recent commit—that checks for file existence and raises a `FileNotFoundError` if the file is missing, preventing unexpected downstream crashes. With a low cyclomatic complexity, this straightforward utility ensures the system fails gracefully during initialization, making it highly maintainable and essential for overall system stability.
### ACTUAL TASK ###
Inputs:
- Code: {code}
- README: {readme}
- Complexity: {complexity}
- Comments: {comments}
- Commits: {commits}
- Code Size: {len(code)}
Output:
"""

def prompt_adv(code, readme, comments, commits, complexity):
    return f"""
You are a senior software engineer with expertise in code readability, maintainability, and program comprehension. Given a source code file along with contextual information, generate a clear and concise code summary. Do not repeat the code. Ensure the summary is optimized for the following assessment aspects: Semantic Correctness, Informativeness, Readability, Usefulness, and Overall Quality.
Follow this step by step thinking process to formulate your response:
Step 1: Analyze the code holistically
- Identify what the code does and determine its main purpose.
- Understand major inputs and outputs (Informativeness).
Step 2: Break down the logic
- Identify key operations and control flow.
- Ensure the technical behavior is understood without misinterpretation (Semantic Correctness).
Step 3: Incorporate contextual information
- Context: Complexity: {complexity}, Comments: {comments}, Commits: {commits}, README: {readme}, Code Size: {len(code)}.
- Use this information to grasp the code's role in system maintenance and its broader context (Usefulness). Do NOT repeat these raw metrics directly.
Step 4: Refine explanation
- Avoid redundancy.
- Use clear, straightforward, and precise language (Readability).
- Ensure the narrative aligns with professional documentation standards (Overall Quality).
Step 5: Generate final summary
- Write a cohesive 2–3 sentence paragraph.
- Focus strictly on functionality and purpose.
Important:
Output ONLY the final summary. Do NOT include your internal steps, reasoning, bullet points, or explanations in the final output.
Code:
{code}
Summary:

"""

def metric_scores(prediction, reference):
    reference_tokens = reference.split()
    prediction_tokens = prediction.split()

    bleu = sentence_bleu(
        [reference_tokens],
        prediction_tokens,
        smoothing_function=smooth,
    )
    rouge_l = rouge_scorer_instance.score(
        reference,
        prediction,
    )["rougeL"].fmeasure
    meteor = meteor_score(
        [reference_tokens],
        prediction_tokens,
    )

    return round(bleu, 6), round(rouge_l, 6), round(meteor, 6)


def prompt_token_count(prompt):
    messages = [
        {
            "role": "system",
            "content": "You are an expert software engineer specializing in code summarization.",
        },
        {"role": "user", "content": prompt},
    ]

    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    return len(
        tokenizer(
            formatted,
            add_special_tokens=True,
            truncation=False,
        )["input_ids"]
    )


def generate_summary(prompt):
    messages = [
        {
            "role": "system",
            "content": "You are an expert software engineer specializing in code summarization.",
        },
        {"role": "user", "content": prompt},
    ]

    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        formatted,
        return_tensors="pt",
        truncation=False,
    ).to(model.device)

    input_length = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated = output[0][input_length:]
    return tokenizer.decode(
        generated,
        skip_special_tokens=True,
    ).strip()


In [ ]:

hf_token = os.getenv("HF_TOKEN")
model_name = "Qwen/Qwen2.5-Coder-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    token=hf_token,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    token=hf_token,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)

if not torch.cuda.is_available():
    model = model.to("cpu")

model.eval()


In [ ]:
records = load_dataset()
results = []

for record in tqdm(records, desc="Qwen"):
    code = record["source_code"]
    readme = record["readme_content"]

    complexity = (
        f"Function Count: {record['complexity']['function_count']}; "
        f"Mean Cyclomatic Complexity: {record['complexity']['mean_ccn']}; "
        f"Maximum Cyclomatic Complexity: {record['complexity']['max_ccn']}; "
        f"NLOC: {record['complexity']['nloc']}"
    )

    comments = record["comments"] or "(none)"

    commits = "\n".join(
        f"{item['hash']} | {item['date']} | {item['subject']}"
        for item in record["commits"]
    ) or "(none)"

    prompts = {
        "Zero": prompt_zero(code, readme, complexity, comments, commits),
        "Few": prompt_few(code, readme, complexity, comments, commits),
        "Adv": prompt_adv(code, readme, comments, commits, complexity),
    }

    token_counts = {
        name: prompt_token_count(prompt)
        for name, prompt in prompts.items()
    }

    max_tokens = max(token_counts.values())

    if max_tokens > MAX_INPUT_TOKENS:
        results.append({
            "File ID": record["file_id"],
            "Repository": record["repository"],
            "Repository SHA": record["repository_sha"],
            "File Name": record["file_name"],
            "File URL": record["file_url"],
            "Skipped": True,
            "Skip Reason": f"Context requires {max_tokens} tokens.",
            "Context Tokens Zero": token_counts["Zero"],
            "Context Tokens Few": token_counts["Few"],
            "Context Tokens Adv": token_counts["Adv"],
        })
        continue

    reference = record["reference_summary"]

    zero_summary = generate_summary(prompts["Zero"])
    few_summary = generate_summary(prompts["Few"])
    adv_summary = generate_summary(prompts["Adv"])

    bz, rz, mz = metric_scores(zero_summary, reference)
    bf, rf, mf = metric_scores(few_summary, reference)
    ba, ra, ma = metric_scores(adv_summary, reference)

    results.append({
        "File ID": record["file_id"],
        "Repository": record["repository"],
        "Repository SHA": record["repository_sha"],
        "File Name": record["file_name"],
        "File URL": record["file_url"],
        "Code Lines": record["code_size"]["lines"],
        "Code Characters": record["code_size"]["characters"],
        "NLOC": record["code_size"]["nloc"],
        "Function Count": record["complexity"]["function_count"],
        "Mean CCN": record["complexity"]["mean_ccn"],
        "Max CCN": record["complexity"]["max_ccn"],
        "Comment Characters": len(record["comments"]),
        "README Characters": len(record["readme_content"]),
        "Commit Count": len(record["commits"]),
        "Reference Summary": reference,
        "Context Tokens Zero": token_counts["Zero"],
        "Context Tokens Few": token_counts["Few"],
        "Context Tokens Adv": token_counts["Adv"],
        "Skipped": False,
        "Skip Reason": "",
        "Zero Summary": zero_summary,
        "BLEU Zero": bz,
        "ROUGE Zero": rz,
        "METEOR Zero": mz,
        "Few Summary": few_summary,
        "BLEU Few": bf,
        "ROUGE Few": rf,
        "METEOR Few": mf,
        "Adv Summary": adv_summary,
        "BLEU Adv": ba,
        "ROUGE Adv": ra,
        "METEOR Adv": ma,
    })

results_df = pd.DataFrame(results)
results_df.to_excel("Qwen_Automatic_Evaluation_Results.xlsx", index=False)
display(results_df.head())
